# Modelagem supervisionada — alfabetização

Prever se o aluno será **alfabetizado** (`alfabetizado` ∈ {0, 1}). É um problema de **classificação binária**.

Este notebook narra a pipeline do Scikit-learn. A execução completa e reproduzível está em `python scripts/run_modeling.py`.


## Candidatos

| Papel | Modelo | Por quê |
|-------|--------|---------|
| Favorito (performance) | Random Forest | não linearidades e interações territoriais / socioeconômicas; SHAP |
| Favorito (explicabilidade) | Regressão logística | citado no enunciado; odds ratio para gestores |
| Baseline | Dummy | piso (~51% acurácia) |
| Didático | Árvore de decisão | regras visíveis |
| Didático | KNN | exige escala + one-hot |

Critério: maior ROC AUC na CV agrupada; empate (< 0,5 pp) → logística.


In [ ]:
from pathlib import Path
import sys
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import GOLD_TABLE, GOLD_YEAR, IMAGES_DIR, RANDOM_STATE, SAMPLE_N, TARGET_COL
from src.preprocessing.features import build_model_frame, feature_lists, xy_groups
from src.preprocessing.load_s3 import load_gold_sample_cached
from src.modeling.split import grouped_holdout, overlap_municipalities, random_holdout
from src.evaluation.leakage import compare_random_vs_grouped
from src.evaluation.metrics import classification_metrics, predict_scores

print(f"tabela={GOLD_TABLE} ano={GOLD_YEAR} SAMPLE_N={SAMPLE_N} seed={RANDOM_STATE}")


## Dados

Amostra aleatória cacheada de `ano=2024`. `build_model_frame` remove constantes/redundâncias e cria `gap_meta`, `dist_uf`, logs.


In [ ]:
raw = load_gold_sample_cached(n=SAMPLE_N, year=GOLD_YEAR, random_state=RANDOM_STATE)
frame = build_model_frame(raw)
X, y, groups = xy_groups(frame)
numeric, low, high = feature_lists(frame)
print(frame.shape, "features", X.shape[1])
print("num", numeric)
print("low", low)
print("high", high)
print(y.value_counts(normalize=True).round(4))
print("municipios", groups.nunique())


## Data leakage

Features preditivas são municipais. Split aleatório coloca o mesmo município nos dois lados.


In [ ]:
leak = compare_random_vs_grouped(X, y, groups)
display(pd.Series(leak))
Xtr_r, Xte_r, ytr_r, yte_r, gtr_r, gte_r = random_holdout(X, y, groups)
Xtr, Xte, ytr, yte, gtr, gte = grouped_holdout(X, y, groups)
print("overlap aleatório", overlap_municipalities(gtr_r, gte_r))
print("overlap agrupado", overlap_municipalities(gtr, gte))


## Pipeline por família

- Linear / KNN: imputação mediana + indicador, StandardScaler, One-Hot (baixa cardinalidade), Target Encoding (alta).
- Árvores: imputação mediana + indicador, Ordinal (baixa), Frequency Encoding (alta), sem scaling.

Tudo dentro de `Pipeline` + `ColumnTransformer`.


In [ ]:
from src.modeling.preprocessing import build_preprocessor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

prep = build_preprocessor(numeric, low, high, family="linear")
pipe = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=1000))])
pipe


## Treino e validação

`scripts/run_modeling.py` treina Dummy, Logística, Árvore, Random Forest e KNN com `RandomizedSearchCV` e `StratifiedGroupKFold`.

Métricas e figuras já geradas pelo script:


In [ ]:
from pathlib import Path
import json

metrics_path = ROOT / "reports" / "model_metrics.json"
search_path = ROOT / "reports" / "model_search.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print("champion:", metrics.get("champion"))
    display(pd.DataFrame(metrics.get("holdout", [])))
else:
    print("Rode: python scripts/run_modeling.py")

for p in sorted((ROOT / "images").glob("model_*.png")):
    print(p.name)


## Interpretabilidade e aplicação

- Permutation importance, coeficientes/odds, SHAP (`images/model_shap_summary.png`).
- Ranking municipal: `reports/risco_municipal.csv`.
- Relatório técnico: `reports/modelagem.md`.


In [ ]:
risco = ROOT / "reports" / "risco_municipal.csv"
if risco.exists():
    df_r = pd.read_csv(risco)
    display(df_r.head(10))
    if "risco_nao_atingir_meta_2024" in df_r.columns:
        print("municipios em risco de não bater a meta 2024 (holdout):", int(df_r["risco_nao_atingir_meta_2024"].sum()))
